# Profiling tests

> **Install for this page:** `pip install scope-profiler pytest`.

scope-profiler ships a pytest plugin that records one region per test, and
`sp.reset()` for test suites that set up the profiler themselves in more than
one test.

In [ ]:
import os
import sys
import tempfile
from pathlib import Path

# Work in a scratch directory so nothing lands in the working directory,
# and make the `scope-profiler` command of this environment available.
WORKDIR = Path(tempfile.mkdtemp(prefix="scope-profiler-guide-"))
os.chdir(WORKDIR)
os.environ["PATH"] = str(Path(sys.executable).parent) + os.pathsep + os.environ["PATH"]
os.environ["PYTHONPATH"] = os.pathsep.join(
    filter(None, [str(WORKDIR), os.environ.get("PYTHONPATH")])
)
sys.path.insert(0, str(WORKDIR))

In [ ]:
%%writefile test_example.py
import time


def test_fast():
    time.sleep(0.001)


def test_slow():
    time.sleep(0.01)

## The pytest plugin

`--scope-profile` turns the plugin on and `--scope-profile-out` names the HDF5
file (default `pytest-profile.h5`). Each test becomes a region
`pytest::<node id>::call`; `--scope-profile-phases all` also records setup and
teardown, and `--scope-profile-config` applies a TOML configuration. The plugin
uses its own profile manager, so tests can still set up and finalize the
global one. It does not work with pytest-xdist (`-n`), because one process must
write the file.

In [ ]:
%%bash
python -m pytest -q -p no:cacheprovider --color=no --scope-profile --scope-profile-out tests.h5 test_example.py
scope-profiler inspect tests.h5 --regions-only --sort total

To see where the time goes inside the tests, enable the opt-in tags of the
libraries they call, for example with `SCOPE_PROFILER_ENABLE_TAGS` (see
[Opt-in tags](/scope-profiler/guide/opt-in-tags/)).

## Resetting between tests

A test that calls `sp.setup` or `sp.session` leaves the profiler configured.
`sp.reset()` forgets the configuration and everything recorded, but keeps the
instrumentation: decorated functions and `define_region` definitions stay
registered and call straight through until the next setup. Use it in an
autouse fixture:

In [ ]:
%%writefile test_reset.py
import pytest

import scope_profiler as sp


@sp.profile("work")
def work():
    return 1


@pytest.fixture(autouse=True)
def _fresh_profiler():
    sp.reset()
    yield
    sp.reset()


@pytest.mark.parametrize("name", ["a.h5", "b.h5"])
def test_profiles(tmp_path, name):
    with sp.session(output=tmp_path / name, verbose=False):
        work()
    regions = [r.name for r in sp.read_h5(tmp_path / name).get_regions()]
    assert "work" in regions

In [ ]:
%%bash
python -m pytest -q -p no:cacheprovider --color=no test_reset.py